DECORATORS

In [ ]:
# decorator is the function that takes function as argument and returns function
# the main use of this decorators are without changing the original function using decorators we can get the desired output

# without arguments
def my_decorator(func):
    def wrapper():
        print("This is wrapper function")
        func()
    return wrapper

# manual decorator
# here my_decorator is decorator
# greet is the function that we are passing to the my_decorator(function)
# returns a function wrapper which catches call
# this is manual representation of decorator

def greet():
    print("Hi")

call=my_decorator(greet)


# @ decorator
# this is the same as above
@my_decorator
def greeting():
    print("hello") 


# calling the decorators
call()
greeting()


In [ ]:
# handling arguments
def my_decorator(func):
    def wrapper(*args,**kwArgs):
        result=func(*args,**kwArgs)
        return result
    return wrapper

@my_decorator
def add(a,b):
    print(a+b)

add(3,4)


In [ ]:
# functools @wraps


# problem

def my_decorator(func):
    def wrapper(*args,**kwArgs):
        '''Hi this is wrapper'''
        result=func(*args,**kwArgs)
        return result
    return wrapper

@my_decorator
def add(a,b):
    '''Hi this is add'''
    print(a+b)

add(3,4)
#help(add)
print(add.__name__) # returns wrapper != add
print(add.__doc__)  # returns wrapper doc != add doc
print()

# solution 
# using @wraps we can reach until add()
from functools import wraps
def my_decorator(func):
    @wraps(func)
    def wrapper(*args,**kwArgs):
        '''Hi this is wrapper'''
        result=func(*args,**kwArgs)
        return result
    return wrapper

@my_decorator
def add(a,b):
    '''Hi this is add'''
    print(a+b)

add(3,4)
#help(add)
print(add.__name__) # returns add
print(add.__doc__)  # returns  add doc

In [ ]:
# decorators with arguments
def decorator(arg):
    def my_decorator(func):
        def wrapper(*args,**kwArgs):
            result=func(*args,**kwArgs)
            print(f"this is {arg}")
            return result
        return wrapper
    return my_decorator


@decorator("info")
def info_message(msg):
    print(f"{msg} from info message")


@decorator("error")
def error_message(msg):
    print(f"{msg} from error message")

info_message("this is info")
error_message("this is error")

In [ ]:
# class decorator
# A. a decorator appplied to a class

def announce(cls):  
    print(f"{cls.__name__} has been created")
    return cls


@announce   # function taking class as argument and returning class
class Person:  
    pass

# here we don't have to call class like function
# decorator will be called when class was created(Person)


# adding method to a class
def announce(cls):
    def bake(self):
        print("Hi this is mahesh")
    cls.bake=bake # adding method to a class
    return cls
    
@announce
class Person:
    pass

p=Person()
p.bake()


# adding __repr__

def announce(cls):
    def __repr__(self):
        print(f"{cls.__name__} : {self.__dict__}")
    cls.__repr__=__repr__
    return cls

@announce
class Person:
    def __init__(self,name,age):
        self.name=name
        self.age=age
print(Person("mahesh",25))

In [ ]:
# keeping the registry
registry=[]
def keep_registry(cls):
    registry.append(cls)
    return cls

@keep_registry
class Person:
    pass

@keep_registry
class Person1:
    pass

@keep_registry
class Person2:
    pass

print(registry)

In [ ]:
# class used as a decorator
from functools import update_wrapper
class Employee:
    def __init__(self,func):
        self.func=func
        update_wrapper(self,func) # this is similar to @wraps in functions
    def __call__(self,*args,**kwArgs):
        result=self.func(*args,**kwArgs)
        print(f"{args} employee has been created in database")
        return result

@Employee
def add_employee(name):
    print("Hi")


add_employee("mahesh")
# for every method call __call__() method will be called


CONTEXT_MANAGERS

In [ ]:
# closing file problem
f=open("file.txt")
data=f.read()
f.close()
# if read() raises any error f.close() never calls, file stays open
# there are 2 ways to solve this problem

# 1.try/finally
f=open("file.txt")
try:
    data=f.read()
finally:
    f.close()


# 2.with
with open("file.txt") as f:
    f.read()
# here using with calls close() function automatically even if read() function raises error
# with calls 2 special methods __enter__() and __exit__()

# checking how thse 2 methods works
class my_context:
    def __enter__(self):
        print("entering") 
    def __exit__(self,exc_type,exc_value,traceback):
        print("leaving")

with my_context() as f:
    print("execute this block")

print("--------------")

class my_context:
    def __enter__(self):
        print("entering") 
    def __exit__(self,exc_type,exc_value,traceback):
        '''
        return False    # (or None) → the exception keeps propagating — normal
        return True      # → the exception is swallowed, silently
        '''
        if exc_type==None:
            print("leaving")
        else:
            print(f"{exc_type}:{exc_value}")

with my_context() as f:
    print("execute this block")
    raise ValueError("oops")

In [62]:
# @contextmanager
# for simple code we don't have to use classes instead we can use @contextmanager
# we have to use generator
from contextlib import contextmanager

@contextmanager
def my_context():
    print("entering")
    yield "some block of code"
    print("exiting")

with my_context() as value: # value which takes yield value
    print(value)


# we must use try finally when error comes otherwise after yield it wont execute next lines of code
print()
@contextmanager
def my_context():
    print("entering")
    try:
        yield ValueError("oops")
    finally:
        print("exiting") # is we dont use try/finally this line will not execute

with my_context() as value:
    print(value)

entering
some block of code
exiting

entering
oops
exiting


In [71]:
# exit stack
with open("file.txt") as f:
    f.read()

# if we have fixed number of files we can open with "with"
# if we have more than 100 files we cannot explicitly call with open(filename)
# so we use exit stack to open and close all 100 files by its own

from contextlib import ExitStack

# generally adding 100 string files to list
files=[]
for i in range(1,3):
    files.append(f"file{i}.txt")

# now working with exitstack
with ExitStack() as stack:
    filestack=[stack.enter_context(open(f"{i}")) for i in files]
    for i in filestack:
        print(i.read())